# পাঠ 11 - মডেল কনটেক্সট প্রটোকল (MCP)

এই **মডেল কনটেক্সট প্রটোকল (MCP)** একটি ওপেন স্ট্যান্ডার্ড যা এজেন্টগুলোকে রানটাইমে ডাইনামিকভাবে টুল, রিসোর্স, এবং ডেটা সোর্স আবিষ্কার ও ব্যবহার করার সক্ষমতা দেয়। একটি এজেন্টে সরাসরি টুলগুলো হার্ডকোড করার বদলে, MCP এজেন্টগুলোকে অন-ডিমান্ডে সক্ষমতা প্রকাশকারী বহিরাগত সার্ভারের সাথে সংযুক্ত হতে দেয়।

In this lesson, you'll learn:
- MCP কী এবং এটি এজেন্ট সিস্টেমগুলোর জন্য কেন গুরুত্বপূর্ণ
- MCP ক্লায়েন্ট-সার্ভার আর্কিটেকচার কীভাবে কাজ করে
- কীভাবে MCP-স্টাইল টুল আবিষ্কার ব্যবহার করে এজেন্ট তৈরি করবেন


## সেটআপ

**প্রয়োজনীয়তা:**
- Azure AI Foundry প্রকল্প যার সাথে একটি ডিপ্লয় করা মডেল রয়েছে
- প্রমাণীকরণের জন্য `az login` চালান


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) কী?

MCP বাইরের টুল এবং ডেটা উৎস আবিষ্কার করা এবং সেগুলোর সাথে ইন্টারঅ্যাক্ট করার জন্য একটি মানসম্মত পদ্ধতি নির্ধারণ করে:

- **MCP Server**: একটি মানসম্মত প্রটোকলের মাধ্যমে টুল, রিসোর্স এবং প্রম্পটগুলো প্রকাশ করে
- **MCP Client**: সার্ভারের সাথে সংযোগ করে এবং উপলব্ধ সক্ষমতাগুলো আবিষ্কার করে এমন এজেন্ট রানটাইম
- **Dynamic Discovery**: এজেন্টদের হার্ডকোড করা টুলের দরকার নেই — তারা রানটাইমে যা উপলব্ধ তা আবিষ্কার করে

এটি এক্সটেনসিবল এজেন্ট সিস্টেম তৈরি করার জন্য শক্তিশালী, যেখানে নতুন সক্ষমতাগুলো এজেন্টের কোড পরিবর্তন না করেই যোগ করা যায়।


## MCP কীভাবে কাজ করে

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. এজেন্ট (MCP ক্লায়েন্ট) একটি MCP সার্ভারের সাথে সংযুক্ত হয়
2. সার্ভার উপলব্ধ টুলগুলির একটি তালিকা এবং তাদের স্কিমাগুলি প্রদান করে
3. এরপর এজেন্ট তার যুক্তি চলাকালীন যে কোনো আবিষ্কৃত টুল কল করতে পারে
4. ফলাফল একই প্রটোকলের মাধ্যমে ফিরে আসে


## MCP টুল আবিষ্কারের অনুকরণ

কারণ একটি বাস্তব MCP সার্ভারের জন্য একটি চলমান সার্ভার প্রক্রিয়া প্রয়োজন, আমরা `@tool` ফাংশনগুলো ব্যবহার করে এই প্যাটার্নটি প্রদর্শন করব যা একটি MCP-সংযুক্ত আবাসন পরিষেবা কী প্রদান করত তা অনুকরণ করে।

প্রোডাকশনে, এই টুলগুলো স্থানীয়ভাবে সংজ্ঞায়িত করার বদলে গতিশীলভাবে একটি MCP সার্ভার থেকে আবিষ্কৃত হতো।


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-স্টাইল টুলস ব্যবহার করে একটি এজেন্ট তৈরি করা


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP প্রোডাকশনে

প্রোডাকশন পরিবেশে, MCP শক্তিশালী প্যাটার্নগুলো সক্ষম করে:

- **ডাইনামিক টুল আবিষ্কার**: এজেন্টগুলি MCP সার্ভারগুলোর সাথে সংযোগ করে এবং রানটাইমে টুল আবিষ্কার করে
- **বিচ্ছিন্ন স্থাপত্য**: টুল প্রদানকারীরা এজেন্ট থেকে আলাদাভাবে আপডেট করতে পারে
- **অর্গানাইজেশন জুড়ে শেয়ারিং**: টিমগুলি MCP সার্ভারের মাধ্যমে সক্ষমতা প্রকাশ করতে পারে যা যেকোনো এজেন্ট ব্যবহার করতে পারে
- **Microsoft Agent Framework সাপোর্ট**: MAF-এ `mcp` ইন্টিগ্রেশনের মাধ্যমে অন্তর্নির্মিত MCP ক্লায়েন্ট সাপোর্ট রয়েছে

MAF-এ একটি বাস্তব MCP সার্ভার ব্যবহার করতে হলে, আপনি `hosted_mcp_tool()` অথবা MCP ক্লায়েন্ট ইন্টিগ্রেশনের মাধ্যমে সংযোগ করবেন।

**আরও জানুন:**
- [MCP স্পেসিফিকেশন](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP সাপোর্ট](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## সারাংশ

এই পাঠে, আপনি শিখেছেন:
- **MCP** হল এজেন্ট এবং টুল প্রদানকারীদের মধ্যে ডাইনামিক টুল আবিষ্কারের জন্য একটি ওপেন স্ট্যান্ডার্ড
- এই **client-server architecture** এজেন্টদের রানটাইমে সক্ষমতা আবিষ্কার করতে দেয়
- MCP সক্ষম করে **প্রসারযোগ্য, বিচ্ছিন্ন এজেন্ট সিস্টেম** যেখানে টুলগুলো কোড পরিবর্তন ছাড়াই যোগ করা যায়
- Microsoft Agent Framework উৎপাদন ব্যবহারের জন্য **অন্তর্নির্মিত MCP সমর্থন** প্রদান করে


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
অস্বীকৃতি:
এই দলিলটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনুবাদ করা হয়েছে। আমরা যতটা সম্ভব সঠিকতা বজায় রাখার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে বলে অনুগ্রহ করে জানুন। স্থানীয় ভাষায় থাকা মূল দলিলটিই কর্তৃত্বপ্রাপ্ত উৎস হিসেবে গণ্য করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদের পরামর্শ দেওয়া হয়। এই অনুবাদ ব্যবহারের ফলে উদ্ভূত কোনো ভুল বোঝাবুঝি বা ভ্রান্ত ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
